In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# %matplotlib inline  # uncomment if running in a notebook

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as T

import albumentations as A
from albumentations.pytorch.transforms import ToTensorV2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



Using device: cuda


In [2]:
# Paths (adjust if necessary)
BASE_PATH = "../input/aerial-cactus-identification"
TRAIN_IMG_PATH = os.path.join(BASE_PATH, "train")
TEST_IMG_PATH = os.path.join(BASE_PATH, "test")
TRAIN_CSV = os.path.join(BASE_PATH, "train.csv")
SAMPLE_SUBMIT = os.path.join(BASE_PATH, "sample_submission.csv")



In [3]:
# Load labels
labels = pd.read_csv(TRAIN_CSV)
labels["has_cactus"] = labels["has_cactus"].astype(int)

# Simple visual check (optional)
# fig = plt.figure(figsize=(12, 6))
# sample_imgs = os.listdir(TRAIN_IMG_PATH)[:20]
# for idx, img_name in enumerate(sample_imgs):
#     ax = fig.add_subplot(4, 5, idx + 1, xticks=[], yticks=[])
#     img = Image.open(os.path.join(TRAIN_IMG_PATH, img_name))
#     ax.imshow(img)
#     ax.set_title(labels.loc[labels['id'] == img_name, 'has_cactus'].values[0])



In [4]:
# Train/validation split
train_df, valid_df = train_test_split(
    labels, stratify=labels["has_cactus"], test_size=0.2, random_state=42
)

# Test dataframe
test_ids = pd.read_csv(SAMPLE_SUBMIT)["id"].values
test_df = pd.DataFrame({"id": test_ids})




In [5]:
class CactusDataset(Dataset):
    def __init__(self, df, img_dir, transforms=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transforms = transforms
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row["id"])
        # read as RGB
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transforms:
            image = self.transforms(image=image)["image"]

        if self.is_test:
            return {"image": image, "id": row["id"]}
        else:
            label = torch.tensor(row["has_cactus"], dtype=torch.float32)
            return {"image": image, "label": label}




In [6]:
def get_train_transforms():
    return A.Compose(
        [
            A.Resize(32, 32),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomBrightnessContrast(p=0.5),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ]
    )


def get_valid_transforms():
    return A.Compose(
        [
            A.Resize(32, 32),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ]
    )




In [7]:
batch_size = 64
num_workers = 0

train_dataset = CactusDataset(
    train_df, TRAIN_IMG_PATH, transforms=get_train_transforms(), is_test=False
)
valid_dataset = CactusDataset(
    valid_df, TRAIN_IMG_PATH, transforms=get_valid_transforms(), is_test=False
)
test_dataset = CactusDataset(
    test_df, TEST_IMG_PATH, transforms=get_valid_transforms(), is_test=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
)




In [8]:
class Net(nn.Module):
    def __init__(self, num_classes=1, dropout_p=0.2):
        super().__init__()
        # Use torchvision's pretrained DenseNet169
        backbone = torchvision.models.densenet169(pretrained=True)
        # Remove original classifier
        self.features = backbone.features
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.flatten = nn.Flatten()
        self.bn = nn.BatchNorm1d(1664)
        self.dropout = nn.Dropout(dropout_p)
        self.fc = nn.Linear(1664, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = self.flatten(x)
        x = self.bn(x)
        x = self.dropout(x)
        x = self.fc(x)
        return x


model = Net(num_classes=1).to(device)



/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet169_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet169_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/densenet169-b2777c0a.pth" to /root/.cache/torch/hub/checkpoints/densenet169-b2777c0a.pth
100%|██████████| 54.7M/54.7M [00:00<00:00, 99.8MB/s]


In [9]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)


def evaluate_auc(loader):
    model.eval()
    all_labels = []
    all_preds = []
    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device)
            labels = batch["label"].cpu().numpy()
            outputs = model(images)
            probs = torch.sigmoid(outputs).cpu().numpy()
            all_labels.extend(labels)
            all_preds.extend(probs)
    return roc_auc_score(all_labels, all_preds)


num_epochs = 5
best_auc = 0.0

for epoch in range(1, num_epochs + 1):
    model.train()
    epoch_losses = []
    for batch in train_loader:
        optimizer.zero_grad()
        images = batch["image"].to(device)
        labels = batch["label"].to(device).unsqueeze(1)  # shape [B,1]
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())
    scheduler.step()
    train_loss = np.mean(epoch_losses)
    val_auc = evaluate_auc(valid_loader)
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), "best_model.pth")
    print(
        f"Epoch {epoch}/{num_epochs} - loss: {train_loss:.4f} - val AUC: {val_auc:.5f} (best {best_auc:.5f})"
    )

# Load best model for inference
model.load_state_dict(torch.load("best_model.pth"))
model.eval()



Epoch 1/5 - loss: 0.0748 - val AUC: 0.99988 (best 0.99988)
Epoch 2/5 - loss: 0.0313 - val AUC: 0.99994 (best 0.99994)
Epoch 3/5 - loss: 0.0186 - val AUC: 0.99996 (best 0.99996)
Epoch 4/5 - loss: 0.0113 - val AUC: 0.99995 (best 0.99996)
Epoch 5/5 - loss: 0.0082 - val AUC: 0.99997 (best 0.99997)


Net(
  (features): Sequential(
    (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu0): ReLU(inplace=True)
    (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (denseblock1): _DenseBlock(
      (denselayer1): _DenseLayer(
        (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (denselayer2): _DenseLayer(
        (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): R

In [10]:
test_preds = []
test_ids = []

with torch.no_grad():
    for batch in test_loader:
        images = batch["image"].to(device)
        ids = batch["id"]
        outputs = model(images)
        probs = torch.sigmoid(outputs).cpu().numpy().reshape(-1)
        test_preds.extend(probs)
        test_ids.extend(ids)



In [11]:
submission = pd.DataFrame({"id": test_ids, "has_cactus": test_preds})
submission.to_csv("sub.csv", index=False)
print("Submission saved to sub.csv")
print(submission.head())

Submission saved to sub.csv
                                     id  has_cactus
0  09034a34de0e2015a8a28dfe18f423f6.jpg    1.000000
1  134f04305c795d6d202502c2ce3578f3.jpg    0.999985
2  41fad8d145e6c41868ce3617e30a2545.jpg    0.999983
3  35f8a11352c8d41b6231bb33d8d09f7e.jpg    0.999960
4  b77dc902b035887cbbc01920ce0e3151.jpg    0.999994
